> ### EEE6503-01: Computer Vision

# Project #3 : Long-tail Visual Recognition for Image Classification

**<div style="text-align: right"> Due date: Jun 23rd, 2026. </div>** 
**<div style="text-align: right"> Please upload your file @ learnus by 11:00 PM. </div>** 

### *Assignment Instructions:*
1. You can use both Korean and English for your report.
2. **Analyze the algorithm, theoretically and empirically.** 
3. **Report your results.**   

<h2><span style="color:blue">[Insert your Group ID HERE]</span> </h2>

In [ ]:
import os
import os.path as osp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
DATASET = 'CIFAR100'

# You should report the results for 4 scenarios
# IMB_TYPE: ['exp', 'step'] X IMB_FACTOR: [0.1, 0.01]
IMB_TYPE = 'exp' #['exp', 'step']
IMB_FACTOR = 0.1 #[0.1, 0.01]

SAVE_DIR = 'logs/exp-0.1'

LR = 0.1
BATCH_SIZE = 128
MOMENTUM = 0.9
WEIGHT_DECAY = 2e-4
EPOCHS = 200

In [ ]:
os.makedirs(osp.join(SAVE_DIR), exist_ok=True)

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

In [ ]:
# ---------------------------------------------------------------------------
# Long-tailed CIFAR datasets (after Cao et al., NeurIPS 2019 / LDAM-DRW)
# ---------------------------------------------------------------------------
class IMBALANCECIFAR10(torchvision.datasets.CIFAR10):
    cls_num = 10

    def __init__(self, root, imb_type='exp', imb_factor=0.01, rand_number=0,
                 train=True, transform=None, target_transform=None,
                 download=False):
        super(IMBALANCECIFAR10, self).__init__(
            root, train, transform, target_transform, download)
        np.random.seed(rand_number)
        img_num_list = self.get_img_num_per_cls(
            self.cls_num, imb_type, imb_factor)
        self.gen_imbalanced_data(img_num_list)

    def get_img_num_per_cls(self, cls_num, imb_type, imb_factor):
        img_max = len(self.data) / cls_num
        img_num_per_cls = []
        if imb_type == 'exp':
            for cls_idx in range(cls_num):
                num = img_max * (imb_factor ** (cls_idx / (cls_num - 1.0)))
                img_num_per_cls.append(int(num))
        elif imb_type == 'step':
            for cls_idx in range(cls_num // 2):
                img_num_per_cls.append(int(img_max))
            for cls_idx in range(cls_num - cls_num // 2):
                img_num_per_cls.append(int(img_max * imb_factor))
        else:
            img_num_per_cls.extend([int(img_max)] * cls_num)
        return img_num_per_cls

    def gen_imbalanced_data(self, img_num_per_cls):
        new_data = []
        new_targets = []
        targets_np = np.array(self.targets, dtype=np.int64)
        classes = np.unique(targets_np)
        self.num_per_cls_dict = dict()
        for the_class, the_img_num in zip(classes, img_num_per_cls):
            self.num_per_cls_dict[the_class] = the_img_num
            idx = np.where(targets_np == the_class)[0]
            np.random.shuffle(idx)
            selec_idx = idx[:the_img_num]
            new_data.append(self.data[selec_idx, ...])
            new_targets.extend([the_class] * the_img_num)
        new_data = np.vstack(new_data)
        self.data = new_data
        self.targets = new_targets

    def get_cls_num_list(self):
        cls_num_list = []
        for i in range(self.cls_num):
            cls_num_list.append(self.num_per_cls_dict[i])
        return cls_num_list


class IMBALANCECIFAR100(IMBALANCECIFAR10):
    """`IMBALANCECIFAR10` with the CIFAR-100 metadata."""
    base_folder = 'cifar-100-python'
    url = "https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz"
    filename = "cifar-100-python.tar.gz"
    tgz_md5 = 'eb9058c3a382ffc7106e4002c42a8d85'
    train_list = [
        ['train', '16019d7e3df5f24257cddd939b257f8d'],
    ]
    test_list = [
        ['test', 'f0ef6b0ae62326f3e7ffdfab6717acfc'],
    ]
    meta = {
        'filename': 'meta',
        'key': 'fine_label_names',
        'md5': '7973b15100ade9c7d40fb424638fde48',
    }
    cls_num = 100


# ---------------------------------------------------------------------------
# Accuracy (overall / head / tail)
# ---------------------------------------------------------------------------
def _topk_accuracy(output, target, topk=(1,)):
    """Standard top-k accuracy (percentage) for the given (output, target)."""
    if target.numel() == 0:
        return [0.0 for _ in topk]
    maxk = max(topk)
    batch_size = target.size(0)
    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0)
        res.append((correct_k * 100.0 / batch_size).item())
    return res


def compute_accuracy(loader, model, topk=(1,)):
    """Evaluate `model` over `loader` and return (overall, head, tail) top-k acc.

    Classes are frequency-sorted by construction in IMBALANCECIFAR (class 0 is
    the most frequent), so the first half of the class indices are treated as
    the head (many-shot) group and the second half as the tail (few-shot) group.
    Each returned value is a list aligned with `topk` (index 0 = top-1).
    """
    was_training = model.training
    model.eval()

    outputs, targets = [], []
    with torch.no_grad():
        for image, target in loader:
            image = image.cuda()
            output = model(image)
            outputs.append(output.detach().cpu())
            targets.append(target.detach().cpu())

    output = torch.cat(outputs, dim=0)
    target = torch.cat(targets, dim=0)

    num_classes = output.size(1)
    split = num_classes // 2
    head_mask = target < split
    tail_mask = target >= split

    topk_acc = _topk_accuracy(output, target, topk)
    head_acc = _topk_accuracy(output[head_mask], target[head_mask], topk)
    tail_acc = _topk_accuracy(output[tail_mask], target[tail_mask], topk)

    if was_training:
        model.train()
    return topk_acc, head_acc, tail_acc


In [ ]:
if DATASET == 'CIFAR10':
    train_dataset = IMBALANCECIFAR10(root='./data', imb_type=IMB_TYPE, imb_factor=IMB_FACTOR, train=True, download=True, transform=transform_train)
    test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
elif DATASET == 'CIFAR100':
    train_dataset = IMBALANCECIFAR100(root='./data', imb_type=IMB_TYPE, imb_factor=IMB_FACTOR, train=True, download=True, transform=transform_train)
    test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

cls_num_list = train_dataset.get_cls_num_list()
print('cls num list:')
print(cls_num_list)
num_classes = len(cls_num_list)

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, drop_last=True)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=100, shuffle=False,
    num_workers=4, )

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, num_blocks):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1,
                               padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        return out  # [B, 512, H, W]

def resnet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])

class ResNet18(nn.Module):
    def __init__(self, num_classes):
        super(ResNet18, self).__init__()
        self.backbone = resnet18()
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        batch_size = x.shape[0]
        x = self.backbone(x)
        x = F.adaptive_max_pool2d(x, 1)
        x = x.view(batch_size, -1)
        pred = self.classifier(x)
        return pred

In [ ]:
model = ResNet18(num_classes)
model = model.cuda()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=150, gamma=0.1)

In [ ]:
for epoch in range(EPOCHS):
    loss_history = []
    model.train()
    for batch_index, data in enumerate(train_loader):
        image, target = data
        image, target = image.cuda(), target.cuda()

        pred = model(image)
        loss = criterion(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())

    topk_acc, head_acc, tail_acc = compute_accuracy(train_loader, model)
    loss_mean = np.mean(loss_history)
    scheduler.step()

    print('Epoch: [{:03d}] \t Loss {:.4f} \t Acc {:.2f} \t AccHead {:.2f} \t AccTail {:.2f}'.format(epoch+1, loss_mean, topk_acc[0], head_acc[0], tail_acc[0]))

torch.save({
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'epoch': epoch},
    osp.join(SAVE_DIR, 'ep{:03d}.pth'.format(epoch+1))
)

In [ ]:
model.eval()
topk_acc, head_acc, tail_acc = compute_accuracy(test_loader, model)

print('Acc {:.2f} \t AccHead {:.2f} \t AccTail {:.2f}'.format(topk_acc[0], head_acc[0], tail_acc[0]))